In [8]:
import pandas as pd
import os

In [ ]:
# Constant map of questions to categories 
ATTENTION_CHECK = 6
REVERSE_CODE = [4, 7, 9, 10, 14, 17]
OVERALL = [1]
COGNITIVE = [2, 3, 4, 5, 7]
AFFECTIVE = [8, 9, 10, 11, 12]
EMOTIONAL = [13, 14, 15, 16, 17]
RESONANCE_COLS = ['Positive Resonance_1', 'Positive Resonance_2', 'Positive Resonance_3']

In [ ]:
# Get the path prefix - for Jupyter notebooks, use os.getcwd() or os.path.dirname(os.getcwd())
start_idx = 2
file_prefix = os.path.join(os.path.dirname(os.getcwd()), 'data')
is_reverse_coded = False
filename = 'final_data-anon'
df = pd.read_csv(f'{file_prefix}/{filename}.csv')
df = df.iloc[start_idx:]
df = df[df['Include'] == 1] # Filter out manually
# Remove Response, Q10, and Q18 columns
df = df
df.head()

348
338


,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,ExternalReference,DistributionChannel,...,Positive Resonance_3,Other_Source_AI_1,Other_Source_Human_1,Age,Race,Race_6_TEXT,Gender,Gender_4_TEXT,Condition,Include
2,2025-12-01 20:03:19,2025-12-01 20:06:42,0,100,203,1,2025-12-01 20:06:43,R_7NDAAwcDRZXIU5b,NaN,anonymous,...,83,10,NaN,1989,4,NaN,1,NaN,Human,1.0
3,2025-12-01 20:03:30,2025-12-01 20:08:38,0,100,307,1,2025-12-01 20:08:38,R_7NIMzbeEVaxJdQq,NaN,anonymous,...,50,12,NaN,1987,1,NaN,2,NaN,Human,1.0
4,2025-12-01 20:03:17,2025-12-01 20:16:48,0,100,811,1,2025-12-01 20:16:48,R_5kjmddY4owG8yvR,NaN,anonymous,...,78,NaN,8,1974,1,NaN,2,NaN,AI,1.0
5,2025-12-01 20:14:13,2025-12-01 20:20:31,0,100,378,1,2025-12-01 20:20:31,R_7fqk2DV0ehoYKPO,NaN,anonymous,...,100,3,NaN,1979,"1,2,3",NaN,2,NaN,Human,1.0
6,2025-12-01 20:13:32,2025-12-01 20:22:55,0,100,563,1,2025-12-01 20:22:55,R_3IzVtLpTKUXza20,NaN,anonymous,...,100,NaN,8,2002,1,NaN,2,NaN,AI,1.0


In [11]:
df.columns

Index(['StartDate', 'EndDate', 'Status', 'Progress', 'Duration (in seconds)',
       'Finished', 'RecordedDate', 'ResponseId', 'ExternalReference',
       'DistributionChannel', 'UserLanguage', 'Consent', 'Prolific ID',
       'Empathy_1', 'Empathy_2', 'Empathy_3', 'Empathy_4', 'Empathy_5',
       'Empathy_6', 'Empathy_7', 'Empathy_8', 'Empathy_9', 'Empathy_10',
       'Empathy_11', 'Empathy_12', 'Empathy_13', 'Empathy_14', 'Empathy_15',
       'Empathy_16', 'Empathy_17', 'Empathy_1.1', 'Empathy_2.1', 'Empathy_3.1',
       'Empathy_4.1', 'Empathy_5.1', 'Empathy_6.1', 'Empathy_7.1',
       'Empathy_8.1', 'Empathy_9.1', 'Empathy_10.1', 'Empathy_11.1',
       'Empathy_12.1', 'Empathy_13.1', 'Empathy_14.1', 'Empathy_15.1',
       'Empathy_16.1', 'Empathy_17.1', 'Positive Resonance_1',
       'Positive Resonance_2', 'Positive Resonance_3', 'Other_Source_AI_1',
       'Other_Source_Human_1', 'Age', 'Race', 'Race_6_TEXT', 'Gender',
       'Gender_4_TEXT', 'Condition', 'Include'],
      dtype=

In [12]:
# Check for duplicate columns and merge emotional_experience columns
def merge_cols_across_conditions(df, column_name):
    # Check if we have both columns
    if column_name in df.columns and f"{column_name}.1" in df.columns:
        df[f'{column_name}'] = df[column_name].fillna(df[f"{column_name}.1"])
        
        # Drop the original duplicate columns
        df = df.drop([f'{column_name}.1'], axis=1)
    return df

def get_column_names(num_range):
    return [f'Empathy_{i}' for i in num_range]

def reverse_code(df, column_name):
    df[column_name] = df[column_name].apply(lambda x: 11 - x)
    return df

columns = ["Emotional_Experience"]
for i in range(1, 18):
    columns.append(f'Empathy_{i}')

for col in columns:
    df = merge_cols_across_conditions(df, col)

# Convert strings to int
for col_name in range(1, 18):
    df[f'Empathy_{col_name}'] = df[f'Empathy_{col_name}'].astype(int)
df[RESONANCE_COLS] = df[RESONANCE_COLS].astype(int)
filtered_df = df
# Filter for Attention Checks 
is_correct = []
for val in df[f'Empathy_{ATTENTION_CHECK}']:
    if val == 10: is_correct.append(1)
    else: is_correct.append(0)
filtered_df['is_correct'] = is_correct

# Reverse Code 
if not is_reverse_coded:
    for col_name in REVERSE_CODE:
        filtered_df = reverse_code(filtered_df, f'Empathy_{col_name}')
    is_reverse_coded = True


# Group by Disaggregated Emotional Category 

cognitive_cols = get_column_names(COGNITIVE)
filtered_df['overall'] = df['Empathy_1']
filtered_df['cognitive'] = filtered_df[cognitive_cols].mean(axis=1)

affective_cols = get_column_names(AFFECTIVE)
filtered_df['affective'] = filtered_df[affective_cols].mean(axis=1)

emotional_cols = get_column_names(EMOTIONAL)
filtered_df['motivational'] = filtered_df[emotional_cols].mean(axis=1)

# Calculate Mean Over Categories 
filtered_df['general_empathy'] = filtered_df[['cognitive', 'affective', 'motivational']].mean(axis=1)

# Calculate Resonance
filtered_df['positive_resonance'] = filtered_df[RESONANCE_COLS].mean(axis=1)
filtered_df['other_source'] = filtered_df['Other_Source_AI_1'].fillna(filtered_df['Other_Source_Human_1'])


filtered_df.to_csv(f'{file_prefix}/{filename}_filtered.csv', index=False)